In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_medication_dosage;
SET search_path TO qdp_medication_dosage;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.medication_dosage',
  '0.1',
  'medication_dosage.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_dosage table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_dosage CASCADE;

CREATE TABLE hub_dosage (
    dosage_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    dosage_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_dosage ADD CONSTRAINT pk_dosage_hubdosage_dosageid PRIMARY KEY (dosage_id);

COMMENT ON COLUMN hub_dosage.dosage_id IS 'Dosage Identity';




-- -----------------------------------------------------------
-- create sat_dosage table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage;

CREATE TABLE sat_dosage (
    sat_dosage_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    dosage_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	sequence INT,
	text VARCHAR,
	patient_instructions VARCHAR,
	timing_id BIGINT,
	as_needed BOOLEAN,
	site_code VARCHAR,
	site_concept_id BIGINT,
	site_raw_code VARCHAR,
	site_raw_concept_id BIGINT,
	route_code VARCHAR,
	route_concept_id BIGINT,
	route_raw_code VARCHAR,
	route_raw_concept_id BIGINT,
	method_code VARCHAR,
	method_concept_id BIGINT,
	method_raw_code VARCHAR,
	method_raw_concept_id BIGINT,
	max_dose_per_administration INT,
	max_dose_per_lifetime INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage ADD CONSTRAINT pk_dosage_satdosage_satdosageid PRIMARY KEY (sat_dosage_id);
ALTER TABLE sat_dosage ADD CONSTRAINT fk_dosage_satdosage_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage.sat_dosage_id IS 'Dosage Identity Sat';
COMMENT ON COLUMN sat_dosage.sequence IS 'The order of the dosage instructions. See https://build.fhir.org/dosage-definitions.html#Dosage.sequence';
COMMENT ON COLUMN sat_dosage.text IS 'Free text dosage instructions e.g. SIG. See https://build.fhir.org/dosage-definitions.html#Dosage.text';
COMMENT ON COLUMN sat_dosage.patient_instructions IS 'Patient or consumer oriented instructions. See https://build.fhir.org/dosage-definitions.html#Dosage.patientInstruction';
COMMENT ON COLUMN sat_dosage.timing_id IS 'When medication should be administered. Reference to the Timing record';
COMMENT ON COLUMN sat_dosage.as_needed IS 'Take "as needed". See https://build.fhir.org/dosage-definitions.html#Dosage.asNeeded';
COMMENT ON COLUMN sat_dosage.site_code IS 'Body site to administer to (SNOMED CT coded). See https://build.fhir.org/dosage-definitions.html#Dosage.site';
COMMENT ON COLUMN sat_dosage.route_code IS 'How drug should enter body (SNOMED CT coded). See https://build.fhir.org/dosage-definitions.html#Dosage.route';
COMMENT ON COLUMN sat_dosage.method_code IS 'Technique for administering medication (SNOMED CT coded). See https://build.fhir.org/dosage-definitions.html#Dosage.method';
COMMENT ON COLUMN sat_dosage.max_dose_per_administration IS 'Upper limit on medication per administration. See https://build.fhir.org/dosage-definitions.html#Dosage.maxDosePerAdministration';
COMMENT ON COLUMN sat_dosage.max_dose_per_lifetime IS 'Upper limit on medication per lifetime of the patient. See https://build.fhir.org/dosage-definitions.html#Dosage.maxDosePerLifetime';



-- -----------------------------------------------------------
-- create sat_dosage_additional_instructions table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage_additional_instructions;

CREATE TABLE sat_dosage_additional_instructions (
	dosage_additional_instruction_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	dosage_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	additional_instruction_code VARCHAR,
	additional_instruction_concept_id BIGINT,
	additional_instruction_raw_code VARCHAR,
	additional_instruction_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage_additional_instructions ADD CONSTRAINT pk_dosage_satdosageadditionalinstr_dosageadditionalinstrid PRIMARY KEY (dosage_additional_instruction_id);
ALTER TABLE sat_dosage_additional_instructions ADD CONSTRAINT fk_dosage_satdosageadditionalinstr_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage_additional_instructions.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage_additional_instructions.dosage_additional_instruction_id IS 'Dosage Additional Instruction Identity';
COMMENT ON COLUMN sat_dosage_additional_instructions.additional_instruction_code IS 'Supplemental instruction or warnings to the patient - e.g. "with meals", "may cause drowsiness" (SNOMED CT coded). See https://hl7.org/fhir/R5/valueset-additional-instruction-codes.html';
COMMENT ON COLUMN sat_dosage_additional_instructions.rank IS 'The recommended order the Additional Instruction Codes should be presented';



-- -----------------------------------------------------------
-- create sat_dosage_as_needed_for table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage_as_needed_for;

CREATE TABLE sat_dosage_as_needed_for (
	dosage_as_needed_for_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	dosage_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	as_needed_for_code VARCHAR,
	as_needed_for_concept_id BIGINT,
	as_needed_for_raw_code VARCHAR,
	as_needed_for_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage_as_needed_for ADD CONSTRAINT pk_dosage_satdosageasneededfor_dosageasneededforid PRIMARY KEY (dosage_as_needed_for_id);
ALTER TABLE sat_dosage_as_needed_for ADD CONSTRAINT fk_dosage_satdosageasneededfor_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage_as_needed_for.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage_as_needed_for.dosage_as_needed_for_id IS 'Dosage As Needed For Identity';
COMMENT ON COLUMN sat_dosage_as_needed_for.as_needed_for_code IS 'Take "as needed" for a specific condition (SNOMED CT clinical findings coded). See https://hl7.org/fhir/R5/valueset-medication-as-needed-reason.html';
COMMENT ON COLUMN sat_dosage_as_needed_for.rank IS 'The recommended order the As Needed For Codes should be presented';



-- -----------------------------------------------------------
-- create sat_dosage_day_of_week table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage_day_of_week;

CREATE TABLE sat_dosage_day_of_week (
	dosage_day_of_week_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	dosage_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	day_of_week_code VARCHAR,
	day_of_week_concept_id BIGINT,
	day_of_week_raw_code VARCHAR,
	day_of_week_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage_day_of_week ADD CONSTRAINT pk_dosage_satdosagedayofweek_dosagedayofweekid PRIMARY KEY (dosage_day_of_week_id);
ALTER TABLE sat_dosage_day_of_week ADD CONSTRAINT fk_dosage_satdosagedayofweek_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage_day_of_week.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage_day_of_week.dosage_day_of_week_id IS 'Dosage Day Of Week Identity';
COMMENT ON COLUMN sat_dosage_day_of_week.day_of_week_code IS 'Day of Week Code. E.g. mon, tue, wed, thu, fri, sat, sun. See https://hl7.org/fhir/R5/valueset-days-of-week.html';
COMMENT ON COLUMN sat_dosage_day_of_week.rank IS 'The recommended order the Day of Week Codes should be presented. By default: mon=1, tue=2, wed=3, thu=4, fri=5, sat=6, sun=7';



-- -----------------------------------------------------------
-- create sat_dosage_dose_and_rate table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage_dose_and_rate;

CREATE TABLE sat_dosage_dose_and_rate (
	dosage_dose_and_rate_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	dosage_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	type_code VARCHAR,
	type_concept_id BIGINT,
	type_raw_code VARCHAR,
	type_raw_concept_id BIGINT,
	dose_quantity INT,
	dose_range_high INT,
	dose_range_low INT,
	rate_quantity INT,
	rate_range_high INT,
	rate_range_low INT,
	rate_ratio_numerator INT,
	rate_ratio_denominator INT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage_dose_and_rate ADD CONSTRAINT pk_dosage_satdosagedoseandrate_dosagedoseandrateid PRIMARY KEY (dosage_dose_and_rate_id);
ALTER TABLE sat_dosage_dose_and_rate ADD CONSTRAINT fk_dosage_satdosagedoseandrate_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage_dose_and_rate.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage_dose_and_rate.dosage_dose_and_rate_id IS 'Dosage Dose And Rate Identity';
COMMENT ON COLUMN sat_dosage_dose_and_rate.type_code IS 'Dose or Rate Type Code. See https://terminology.hl7.org/5.1.0/ValueSet-dose-rate-type.html';
COMMENT ON COLUMN sat_dosage_dose_and_rate.dose_quantity IS 'Amount of medication per dose. See https://build.fhir.org/dosage-definitions.html#Dosage.doseAndRate.dose_x_';
COMMENT ON COLUMN sat_dosage_dose_and_rate.dose_range_high IS 'High end of the amount of medication per dose range';
COMMENT ON COLUMN sat_dosage_dose_and_rate.dose_range_low IS 'Low end of the amount of medication per dose range';
COMMENT ON COLUMN sat_dosage_dose_and_rate.rate_quantity IS 'Amount of medication per unit of time. See https://build.fhir.org/dosage-definitions.html#Dosage.doseAndRate.rate_x_';
COMMENT ON COLUMN sat_dosage_dose_and_rate.rate_range_high IS 'High end of the amount of medication per unit of time range';
COMMENT ON COLUMN sat_dosage_dose_and_rate.rate_range_low IS 'Low end of the amount of medication per unit of time range';
COMMENT ON COLUMN sat_dosage_dose_and_rate.rate_ratio_numerator IS 'Rate ratio numerator - amount of medication per unit of time';
COMMENT ON COLUMN sat_dosage_dose_and_rate.rate_ratio_denominator IS 'Rate ratio denominator - amount of medication per unit of time';
COMMENT ON COLUMN sat_dosage_dose_and_rate.rank IS 'The recommended order the Dose And Rate entries should be presented';



-- -----------------------------------------------------------
-- create sat_dosage_max_dose_per_period table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage_max_dose_per_period;

CREATE TABLE sat_dosage_max_dose_per_period (
	dosage_max_dose_per_period_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	dosage_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	rate_ratio_numerator INT,
	rate_ratio_denominator INT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage_max_dose_per_period ADD CONSTRAINT pk_dosage_satdosagemaxdoseperperiod_dosagemaxdoseperperiodid PRIMARY KEY (dosage_max_dose_per_period_id);
ALTER TABLE sat_dosage_max_dose_per_period ADD CONSTRAINT fk_dosage_satdosagemaxdoseperperiod_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage_max_dose_per_period.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage_max_dose_per_period.dosage_max_dose_per_period_id IS 'Dosage Max Dose Per Period Identity';
COMMENT ON COLUMN sat_dosage_max_dose_per_period.rate_ratio_numerator IS 'Upper limit on medication per unit of time (numerator). See https://build.fhir.org/dosage-definitions.html#Dosage.maxDosePerPeriod';
COMMENT ON COLUMN sat_dosage_max_dose_per_period.rate_ratio_denominator IS 'Upper limit on medication per unit of time (denominator)';
COMMENT ON COLUMN sat_dosage_max_dose_per_period.rank IS 'The recommended order the Max Dose Per Period entries should be presented';



-- -----------------------------------------------------------
-- create sat_dosage_time_of_day table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage_time_of_day;

CREATE TABLE sat_dosage_time_of_day (
	dosage_time_of_day_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	dosage_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	time VARCHAR,
	display VARCHAR,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage_time_of_day ADD CONSTRAINT pk_dosage_satdosagetimeofday_dosagetimeofdayid PRIMARY KEY (dosage_time_of_day_id);
ALTER TABLE sat_dosage_time_of_day ADD CONSTRAINT fk_dosage_satdosagetimeofday_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage_time_of_day.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage_time_of_day.dosage_time_of_day_id IS 'Dosage Time Of Day Identity';
COMMENT ON COLUMN sat_dosage_time_of_day.time IS 'A time during the day, in the format hh:mm:ss. Regex: ([01][0-9]|2[0-3]):[0-5][0-9]:([0-5][0-9]|60). See https://hl7.org/fhir/R5/datatypes.html#time';
COMMENT ON COLUMN sat_dosage_time_of_day.display IS 'The descriptive text to be used with the Time Of Day';
COMMENT ON COLUMN sat_dosage_time_of_day.rank IS 'The recommended order the Time Of Day entries should be presented';



-- -----------------------------------------------------------
-- create sat_dosage_when table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage_when;

CREATE TABLE sat_dosage_when (
	dosage_when_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	dosage_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	when_code VARCHAR,
	when_concept_id BIGINT,
	when_raw_code VARCHAR,
	when_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage_when ADD CONSTRAINT pk_dosage_satdosagewhen_dosagewhenid PRIMARY KEY (dosage_when_id);
ALTER TABLE sat_dosage_when ADD CONSTRAINT fk_dosage_satdosagewhen_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage_when.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage_when.dosage_when_id IS 'Dosage When Identity';
COMMENT ON COLUMN sat_dosage_when.when_code IS 'When during the day the dosage should occur. E.g. MORN (Morning), MORN.early, MORN.late, NOON, AFT (Afternoon), AFT.early, AFT.late, EVE (Evening), EVE.early, EVE.late, NIGHT, PHS (After Sleep), IMD (Immediate). See https://hl7.org/fhir/R5/valueset-event-timing.html';
COMMENT ON COLUMN sat_dosage_when.rank IS 'The recommended order the Dosage When Codes should be presented';



-- -----------------------------------------------------------
-- create sat_dosage_timing table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_dosage_timing;

CREATE TABLE sat_dosage_timing (
	dosage_timing_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	dosage_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	timing_code VARCHAR,
	timing_code_concept_id BIGINT,
	timing_code_raw_code VARCHAR,
	timing_code_raw_concept_id BIGINT,
	bounds_duration_value DECIMAL,
	bounds_duration_time_period VARCHAR,
	bounds_range_high DECIMAL,
	bounds_duration_low DECIMAL,
	bounds_period_start TIMESTAMP,
	bounds_period_end TIMESTAMP,
	count INT,
	count_max INT,
	duration DECIMAL,
	duration_max DECIMAL,
	duration_unit VARCHAR,
	duration_unit_concept_id BIGINT,
	duration_unit_raw_code VARCHAR,
	duration_unit_raw_concept_id BIGINT,
	frequency INT,
	frequency_max INT,
	frequency_period DECIMAL,
	frequency_period_max DECIMAL,
	frequency_period_unit VARCHAR,
	frequency_period_unit_concept_id BIGINT,
	frequency_period_unit_raw_code VARCHAR,
	frequency_period_unit_raw_concept_id BIGINT,
	timing_offset INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_dosage_timing ADD CONSTRAINT pk_dosage_satdosagetiming_dosagetimingid PRIMARY KEY (dosage_timing_id);
ALTER TABLE sat_dosage_timing ADD CONSTRAINT fk_dosage_satdosagetiming_hubdosage_dosageid FOREIGN KEY (dosage_id) REFERENCES hub_dosage(dosage_id);

COMMENT ON COLUMN sat_dosage_timing.dosage_id IS 'Dosage Identity';
COMMENT ON COLUMN sat_dosage_timing.dosage_timing_id IS 'Dosage Timing Identity';
COMMENT ON COLUMN sat_dosage_timing.timing_code IS 'Abbreviated Timing Code. E.g. BID (twice daily), TID (three times daily), QID (four times daily), AM, PM, QD (every day), QOD (every other day), Q1H-Q8H (every N hours), BED (bedtime), WK (weekly), MO (monthly). See https://hl7.org/fhir/R5/valueset-timing-abbreviation.html';
COMMENT ON COLUMN sat_dosage_timing.bounds_duration_value IS 'Length/Range of lengths, or (Start and/or end) limits. See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.bounds_x_';
COMMENT ON COLUMN sat_dosage_timing.bounds_duration_time_period IS 'Unit of bounds duration. E.g. Second, Minute, Hour, Day, Month, Year';
COMMENT ON COLUMN sat_dosage_timing.bounds_range_high IS 'High end of the bounds range';
COMMENT ON COLUMN sat_dosage_timing.bounds_duration_low IS 'Low end of the bounds range';
COMMENT ON COLUMN sat_dosage_timing.bounds_period_start IS 'Start of bounds period. See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.bounds_x_';
COMMENT ON COLUMN sat_dosage_timing.bounds_period_end IS 'End of bounds period';
COMMENT ON COLUMN sat_dosage_timing.count IS 'Number of times to repeat. See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.count';
COMMENT ON COLUMN sat_dosage_timing.count_max IS 'Maximum number of times to repeat. See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.countMax';
COMMENT ON COLUMN sat_dosage_timing.duration IS 'How long when it happens. See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.duration';
COMMENT ON COLUMN sat_dosage_timing.duration_max IS 'How long when it happens (Max). See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.durationMax';
COMMENT ON COLUMN sat_dosage_timing.duration_unit IS 'Unit of time (UCUM). E.g. s (second), min (minute), h (hour), d (day), wk (week), mo (month), a (year). See http://unitsofmeasure.org';
COMMENT ON COLUMN sat_dosage_timing.frequency IS 'Number of repetitions within a period. See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.frequency';
COMMENT ON COLUMN sat_dosage_timing.frequency_max IS 'Maximum number of repetitions within a period. See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.frequencyMax';
COMMENT ON COLUMN sat_dosage_timing.frequency_period IS 'The duration to which the frequency applies. See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.period';
COMMENT ON COLUMN sat_dosage_timing.frequency_period_max IS 'Upper limit of period (e.g. 3-4 hours). See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.periodMax';
COMMENT ON COLUMN sat_dosage_timing.frequency_period_unit IS 'Unit of time for frequency period (UCUM). E.g. s (second), min (minute), h (hour), d (day), wk (week), mo (month), a (year). See http://unitsofmeasure.org';
COMMENT ON COLUMN sat_dosage_timing.timing_offset IS 'Minutes from event (before or after). See https://build.fhir.org/datatypes-definitions.html#Timing.repeat.offset';




-- -----------------------------------------------------------
-- create view_dosages view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_dosages AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.dosage_id,
	h.dosage_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_dosage_id,
  s.sequence,
  s.text,
  s.patient_instructions,
  s.timing_id,
  s.as_needed,
  s.site_code,
  s.site_concept_id,
  s.site_raw_code,
  s.site_raw_concept_id,
  s.route_code,
  s.route_concept_id,
  s.route_raw_code,
  s.route_raw_concept_id,
  s.method_code,
  s.method_concept_id,
  s.method_raw_code,
  s.method_raw_concept_id,
  s.max_dose_per_administration,
  s.max_dose_per_lifetime,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_dosage.hub_dosage h
    JOIN budd.qdp_medication_dosage.sat_dosage s
      ON h.dosage_id = s.dosage_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_dosages;


-- -----------------------------------------------------------
-- create view_dosage_dose_and_rates view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_dosage_dose_and_rates AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.dosage_id,
	h.dosage_entity_id,
  s.text,
  s.route_code,
  s.method_code,
	dr.dosage_dose_and_rate_id,
	dr.type_code,
	dr.type_concept_id,
	dr.type_raw_code,
	dr.type_raw_concept_id,
	dr.dose_quantity,
	dr.dose_range_high,
	dr.dose_range_low,
	dr.rate_quantity,
	dr.rate_range_high,
	dr.rate_range_low,
	dr.rate_ratio_numerator,
	dr.rate_ratio_denominator,
	dr.rank AS dose_and_rate_rank,
	dr.period_start AS dose_and_rate_period_start,
	dr.period_end AS dose_and_rate_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_dosage.hub_dosage h
    JOIN budd.qdp_medication_dosage.sat_dosage s
      ON h.dosage_id = s.dosage_id
    JOIN budd.qdp_medication_dosage.sat_dosage_dose_and_rate dr
      ON h.dosage_id = dr.dosage_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_dosage_dose_and_rates;


-- -----------------------------------------------------------
-- create view_dosage_schedules view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_dosage_schedules AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.dosage_id,
	h.dosage_entity_id,
  s.text,
  s.as_needed,
	dw.dosage_day_of_week_id,
	dw.day_of_week_code,
	dw.day_of_week_concept_id,
	dw.day_of_week_raw_code,
	dw.day_of_week_raw_concept_id,
	dw.rank AS day_of_week_rank,
	dw.period_start AS day_of_week_period_start,
	dw.period_end AS day_of_week_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_dosage.hub_dosage h
    JOIN budd.qdp_medication_dosage.sat_dosage s
      ON h.dosage_id = s.dosage_id
    JOIN budd.qdp_medication_dosage.sat_dosage_day_of_week dw
      ON h.dosage_id = dw.dosage_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_dosage_schedules;


-- -----------------------------------------------------------
-- create view_dosage_timings view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_dosage_timings AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.dosage_id,
	h.dosage_entity_id,
  s.text,
  s.sequence,
	dt.dosage_timing_id,
	dt.timing_code,
	dt.timing_code_concept_id,
	dt.timing_code_raw_code,
	dt.timing_code_raw_concept_id,
	dt.bounds_duration_value,
	dt.bounds_duration_time_period,
	dt.bounds_range_high,
	dt.bounds_duration_low,
	dt.bounds_period_start,
	dt.bounds_period_end,
	dt.count,
	dt.count_max,
	dt.duration,
	dt.duration_max,
	dt.duration_unit,
	dt.frequency,
	dt.frequency_max,
	dt.frequency_period,
	dt.frequency_period_max,
	dt.frequency_period_unit,
	dt.timing_offset,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_dosage.hub_dosage h
    JOIN budd.qdp_medication_dosage.sat_dosage s
      ON h.dosage_id = s.dosage_id
    JOIN budd.qdp_medication_dosage.sat_dosage_timing dt
      ON h.dosage_id = dt.dosage_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_dosage_timings;


/**

**/
